# 제주 C-ITS·ITS 인증키·실시간성 감사 v0.1

키 값을 읽거나 표시하지 않고 저장된 감사 결과의 인증, 레코드 수, 실시간성, 링크 결합률을 재검토한다.

In [1]:
import json
from pathlib import Path

project_root = Path.cwd()
if project_root.name == '03_notebooks':
    project_root = project_root.parent
report_path = project_root / '02_outputs/data/jeju_live_api_audit_v0_1/audit_report.json'
report = json.loads(report_path.read_text(encoding='utf-8'))
{
    'status': report['status'],
    'evaluated_at': report['evaluated_at'],
    'credential_checks': report['credential_checks'],
    'realtime_traffic_status': report['realtime_traffic_status'],
    'failure_count': report['failure_count'],
}


{'status': 'pass_with_limitations',
 'evaluated_at': '2026-10-27T09:21:29.881241+09:00',
 'credential_checks': {'JEJU_CITS_API_KEY': {'status': 'valid',
   'validated_source_count': 4,
   'validated_sources': ['jeju_cits_road_event',
    'jeju_cits_road_close',
    'jeju_cits_road_work',
    'jeju_cits_road_weather']},
  'JEJU_ITS_API_KEY': {'status': 'valid',
   'validated_source_count': 3,
   'validated_sources': ['jeju_its_realtime_traffic',
    'jeju_its_hourly_traffic',
    'jeju_its_daily_traffic']}},
 'realtime_traffic_status': 'fresh',
 'failure_count': 0}

In [2]:
summary = []
for source in report['sources']:
    quality = source['quality']
    summary.append({
        'source_id': source['source_id'],
        'records': source['records']['extracted_count'],
        'authentication': source['authentication']['status'],
        'timeliness': source['timeliness']['status'],
        'latest_age_minutes': source['timeliness'].get('latest_age_minutes'),
        'exact_join_rate': quality['summary']['valid_id_exact_join_rate'],
        'failed_check_count': len(quality['failed_checks']),
    })
summary


[{'source_id': 'jeju_cits_road_event',
  'records': 33,
  'authentication': 'valid',
  'timeliness': 'not_verifiable',
  'latest_age_minutes': None,
  'exact_join_rate': 0.727273,
  'failed_check_count': 1},
 {'source_id': 'jeju_cits_road_close',
  'records': 0,
  'authentication': 'valid',
  'timeliness': 'not_verifiable',
  'latest_age_minutes': None,
  'exact_join_rate': None,
  'failed_check_count': 12},
 {'source_id': 'jeju_cits_road_work',
  'records': 1,
  'authentication': 'valid',
  'timeliness': 'not_verifiable',
  'latest_age_minutes': None,
  'exact_join_rate': 0.0,
  'failed_check_count': 1},
 {'source_id': 'jeju_cits_road_weather',
  'records': 12,
  'authentication': 'valid',
  'timeliness': 'not_verifiable',
  'latest_age_minutes': None,
  'exact_join_rate': None,
  'failed_check_count': 2},
 {'source_id': 'jeju_its_realtime_traffic',
  'records': 12081,
  'authentication': 'valid',
  'timeliness': 'fresh',
  'latest_age_minutes': 1.498,
  'exact_join_rate': 0.985514,
 

In [3]:
realtime = next(row for row in summary if row['source_id'] == 'jeju_its_realtime_traffic')
assert realtime['authentication'] == 'valid'
assert realtime['timeliness'] == 'fresh'
assert realtime['latest_age_minutes'] <= 30
assert report['failure_count'] == 0
'Credential and realtime gates passed; C-ITS timestamp limitations remain.'


'Credential and realtime gates passed; C-ITS timestamp limitations remain.'